# Build, audit, and repair a C4 qodec

Start with a code's stabilizers and logical operators, then turn them into a protocol we can inspect and test. The first build has incomplete measurement equations and circuits that lose the code's distance. We will repair both, keeping all six instructions that the builder can implement.

`qodec` holds the code, instruction definitions, circuits, and measurement equations. `qdk.ec` builds circuits and checks their consequences. Our endpoint is a qodec with **no audit diagnostics** and **no undetected single-fault logical failure under the fault model below**. These are separate checks: correct noiseless behavior does not guarantee fault tolerance.

## Installing

```bash
pip install "qdk[ec]"
```

## 1. Start with the C4 code

The $[[4,2,2]]$ code stores two logical qubits in four physical qubits. Its two stabilizers are $XXXX$ and $ZZZZ$. The `x` and `z` lists below choose the logical operators and their order.

In [1]:
from collections import Counter
from pathlib import Path

from IPython.display import Markdown, display
import qodec as qc
import qdk.ec as ec

c4 = qc.Code(
    "C4",
    stabilizers=["X_0 X_1 X_2 X_3", "Z_0 Z_1 Z_2 Z_3"],
    x=["X_0 X_1", "X_0 X_2"],
    z=["Z_0 Z_2", "Z_0 Z_1"],
)

### Check the code distance

`SubsystemCode` supplies algebraic analysis of the code definition. Its distance search returns a minimum set of single-qubit Pauli errors that preserve the stabilizers but change the encoded information. Multiply the witness factors to see that logical error.

In [2]:
code = ec.SubsystemCode.of(c4)
code_distance, code_witness = code.distance()
logical_error = ec.Pauli.identity()
for factor in code_witness:
    logical_error *= factor

print("Code distance:", code_distance)
print("Witness factors:", [str(factor) for factor in code_witness])
print("Combined data error:", logical_error)
print("Syndrome:", sorted(code.syndrome_of(logical_error)))
print("Logical effect:", code.logical_effect_of(logical_error))
assert code_distance == logical_error.weight == 2
assert not code.syndrome_of(logical_error)
assert code.logical_effect_of(logical_error).weight > 0

Code distance: 2
Witness factors: ['X', 'IX']
Combined data error: XX
Syndrome: []
Logical effect: X


The code detects any single-qubit data error, but it cannot correct every such error without extra information. A circuit fault can spread to several data qubits, so the code distance does not certify the gadgets.

## 2. Build and explore the qodec

Build a qodec directly from this definition with `strategy="bare-css/v1"`. This construction uses syndrome ancillas but **no flag qubits**, and is not fault tolerant. `strict=False` keeps candidates that pass completion and logical-action verification, and records the ones it cannot build. It does not bypass those checks or promise a clean audit.

A qodec is an ordered stack of layers. Each layer has an instruction set; its gadgets implement those instructions using circuits over the next layer. This build has one encoded C4 layer and one physical `stim` layer.

An instruction describes the logical operation. A gadget adds its circuit, input/output encodings, checks, and logical readouts. One encoded block contains four physical qubits here; `transversal_cx` takes two blocks.

Inspect the instruction menu and the omission record. Transversal H fails for this logical basis because it does not act as H on each logical qubit. It is absent from both the instruction set and the gadgets. Logical Paulis are frame updates, not circuit candidates.

In [3]:
protocol = ec.build_qodec(c4, strategy="bare-css/v1", strict=False)
layer = protocol.layers[0]
instruction_menu = set(layer.instruction_set.instructions)
omitted = protocol.metadata["qdk.ec"]["build"]["omitted"]

assert protocol.metadata["qdk.ec"]["build"]["flags_per_stabilizer"] == 0
assert instruction_menu == set(layer.gadgets) == {
    "prepare_x", "prepare_z", "idle", "measure_x", "measure_z", "transversal_cx"
}
assert set(omitted) == {"transversal_h"}
display(protocol, layer, omitted)

Qodec "C4"
  Built from the 'C4' stabilizer code ([[4, 2]]). Strategy: bare-css/v1.
  Layers: C4 -> stim
  Lowering:
    C4 -> stim: 6 gadgets (idle, measure_x, measure_z, prepare_x, prepare_z, ...+1)
  Codes: C4

Layer "C4"
  Instructions: 6
  Gadgets: 6 (idle, measure_x, measure_z, prepare_x, prepare_z, transversal_cx)

{'transversal_h': {'kind': 'ActionMismatch',
  'message': 'logical action differs between declared and realized',
  'stage': 'verification'}}

### Inspect the idle gadget

`idle` should preserve both logical qubits while measuring the two stabilizers. Its input and output encodings connect the logical block to circuit qubits 0-3; qubits 4 and 5 are syndrome ancillas.

Inspect the instruction and gadget directly below. Their YAML displays show the declarations without running analysis. The gadget snippet relies on its containing layer for the instruction-set and code bindings; it is not a self-contained bundle.

Checks are parity equations expected to be zero without faults. They can refer to circuit measurement bits and to encoding signs at either boundary. `GadgetProfile.objective` describes the declared instruction; `action` describes what the circuit actually does.

In [4]:
bare_idle = layer.gadgets["idle"]
bare_idle

circuit:
  source: |
    R 4
    H 4
    CX 4 0
    CX 4 1
    CX 4 2
    CX 4 3
    H 4
    R 5
    H 5
    CZ 5 0
    CZ 5 1
    CZ 5 2
    CZ 5 3
    H 5
    M 4 5
  format: stim
  in:
    '0': qubit
    '1': qubit
    '2': qubit
    '3': qubit
  out:
    '0': qubit
    '1': qubit
    '2': qubit
    '3': qubit
in:
- C4: [0, 1, 2, 3]
out:
- C4: [0, 1, 2, 3]
checks:
- ['circuit.readouts[0]', 'in[0].stabilizers[0]']
- ['circuit.readouts[1]', 'in[0].stabilizers[1]']
- ['circuit.readouts[0]', 'out[0].stabilizers[0]']
- ['circuit.readouts[1]', 'out[0].stabilizers[1]']


In [5]:
bare_idle_profile = ec.GadgetProfile(bare_idle)
objective = bare_idle_profile.objective
assert objective is not None
assert bare_idle_profile.action.is_equivalent_to(objective)

## 3. Audit the declarations

A circuit can have the right logical action while its measurement equations report the wrong answer. Run `ec.audit` on the whole qodec and inspect one error and one warning.

This build has four incorrect logical-readout equations and four missing output-frame relations. These are limitations of the builder's declarations, not failed candidates retained by `strict=False`. The failed H candidate was already removed.

The qodec was built in memory, so the diagnostics identify model paths rather than source-file locations.

In [6]:
initial_report = ec.audit(protocol)
counts = Counter((item.severity.name, item.rule) for item in initial_report.diagnostics)
display(Markdown("\n".join([
    "| Severity | Rule | Count |",
    "| --- | --- | ---: |",
    *(f"| {severity} | `{rule}` | {count} |"
      for (severity, rule), count in sorted(counts.items())),
])))
print(ec.Report((initial_report.errors[0], initial_report.warnings[0])))
assert counts == {
    ("ERROR", "gadget/readout-mismatch"): 4,
    ("WARNING", "gadget/incomplete-output-frame"): 4,
}

| Severity | Rule | Count |
| --- | --- | ---: |
| ERROR | `gadget/readout-mismatch` | 4 |
| WARNING | `gadget/incomplete-output-frame` | 4 |

[ERROR] gadget/readout-mismatch
layers[0].gadgets['measure_x'] (C4 -> stim)
readouts[0] does not report the required logical X_0 measurement
    Declared equation: ["circuit.readouts[0]", "circuit.readouts[1]"]
    Verified readout equation: ["in[0].x[0]", "circuit.readouts[0]", "circuit.readouts[1]"]

[WARNING] gadget/incomplete-output-frame
layers[0].gadgets['transversal_cx'] (C4 -> stim)
Declared relations do not determine out[0].stabilizers[0] (X_0 X_1 X_2 X_3)
    Code: C4; circuit support: ['0', '1', '2', '3'].
    Verified relation: ["out[0].stabilizers[0]", "in[0].stabilizers[0]", "in[1].stabilizers[0]"]

audit: 1 error(s), 1 warning(s), 0 informational


### Repair the logical readouts

A Pauli frame records a known correction without applying physical Pauli gates. It can change the sign of a logical measurement. The builder's readouts use only circuit bits, so they work for a zero incoming frame but not for every frame.

For logical X readout `position`, add `in[0].x[position]`; for Z, add `in[0].z[position]`. These are the missing terms shown by the audit. The circuit itself does not change.

We can keep both measurement instructions by fixing their equations. Removing an unsupported operation would instead require removing both its instruction and its gadget.

In [7]:
for basis in ("x", "z"):
    measurement = layer.gadgets[f"measure_{basis}"]
    measurement.readouts = [
        [*readout.equation, f"in[0].{basis}[{position}]"]
        for position, readout in enumerate(measurement.readouts)
    ]

readout_report = ec.audit(protocol)
print("After readout repair:", len(readout_report.errors), "errors,",
      len(readout_report.warnings), "warnings")
assert not readout_report.errors
assert len(readout_report.warnings) == 4

After readout repair: 0 errors, 4 warnings


### Complete the CNOT output frames

The remaining warnings concern `transversal_cx`. Its circuit implements the right logical CNOT, but its declarations do not say how the input stabilizer signs determine the output signs.

Stabilizer index 0 is X-type and index 1 is Z-type. Under CNOT, the first output's X stabilizer pulls back to the product of both input X stabilizers; the second output's Z stabilizer pulls back to both input Z stabilizers. The other two signs pass through. Write those four relations as zero-parity equations, as suggested by the audit.

These checks add no gates, ancillas, or physical measurement outcomes. They complete the boundary-frame bookkeeping.

In [8]:
layer.gadgets["transversal_cx"].checks = [
    ["out[0].stabilizers[0]", "in[0].stabilizers[0]", "in[1].stabilizers[0]"],
    ["out[0].stabilizers[1]", "in[0].stabilizers[1]"],
    ["out[1].stabilizers[0]", "in[1].stabilizers[0]"],
    ["out[1].stabilizers[1]", "in[0].stabilizers[1]", "in[1].stabilizers[1]"],
]
correct_report = ec.audit(protocol)
print(correct_report)
assert not correct_report.diagnostics
assert set(layer.instruction_set.instructions) == set(layer.gadgets) == instruction_menu

audit: ok (no diagnostics)


## 4. Measure the gadget distances

The declarations now pass every audit rule, including the output-frame checks. That establishes noiseless correctness under the supported model, not resistance to faults.

**Code distance counts data errors; gadget distance counts circuit faults.** `GadgetProfile.distance()` finds the fewest allowed faults that leave the declared checks unchanged, preserve the output codespaces, and change the logical action. It returns a distance and a list of fault events witnessing that failure.

The default model combines post-call Pauli errors with flips of the readout bits produced by that call. Each event costs one, even when several qubits and readouts are affected together. Calls without readouts have three one-qubit or fifteen two-qubit Pauli errors.

A readout flip changes the recorded result without changing the surviving quantum state. For a non-destructive Pauli measurement, this is equivalent to an anticommuting Pauli before and after the measurement. `ec.FaultEvent.after(7, readout_flips=0)` flips the first readout produced by call 7; `[0, 2]` would select its first and third readouts. The same constructor accepts a Pauli error, a readout change, or both.

Run the exact search without a cutoff for every gadget. Each distance below has a replayable witness; the Z-measurement witness is displayed after the table.

In [9]:
before_distances = {
    mnemonic: ec.GadgetProfile(gadget).distance()
    for mnemonic, gadget in layer.gadgets.items()
}
assert all(gadget_distance == len(witness) for gadget_distance, witness in before_distances.values())
display(Markdown("\n".join([
    "| Gadget | Circuit fault distance | Witness events |",
    "| --- | ---: | ---: |",
    *(f"| `{mnemonic}` | {gadget_distance} | {len(witness)} |"
      for mnemonic, (gadget_distance, witness) in sorted(before_distances.items())),
])))
low_distance = {
    mnemonic for mnemonic, (gadget_distance, witness) in before_distances.items()
    if gadget_distance < code_distance
}
assert low_distance == {"idle", "prepare_x", "prepare_z"}
measure_z_distance, measure_z_witness = before_distances["measure_z"]
assert measure_z_distance == len(measure_z_witness) == code_distance
readout_faults = {
    ec.FaultEvent.after(index, readout_flips=0)
    for index in range(len(layer.gadgets["measure_z"].circuit.calls))
}
assert all(fault in readout_faults for fault in measure_z_witness)
display(measure_z_witness)

| Gadget | Circuit fault distance | Witness events |
| --- | ---: | ---: |
| `idle` | 1 | 1 |
| `measure_x` | 2 | 2 |
| `measure_z` | 2 | 2 |
| `prepare_x` | 1 | 1 |
| `prepare_z` | 1 | 1 |
| `transversal_cx` | 2 | 2 |

[FaultEvent.after(0, readout_flips=0), FaultEvent.after(1, readout_flips=0)]

`measure_z` has distance two: one recorded-bit flip triggers its parity check, but two can cancel in that check while changing a logical result. Destructive measurements are noisy even though they leave no quantum output.

### Trace the idle failure

A single X fault on ancilla 4 after `CX 4 1` spreads to data error $X_2X_3$. The final H on ancilla 4 turns its remaining X into Z, leaving its Z-basis measurement unchanged. The two data X errors affect the later Z-syndrome ancilla twice, cancelling. The data error is logical, yet all checks stay zero.

Each witness is displayed as a replayable `FaultEvent.after(...)` expression. The first argument always indexes `Circuit.calls` from zero, not source-line numbers; any readout indexes are local to that call. Multiply the events and pass their product to `effects_of` to replay the combined effect without inspecting their storage.

In [10]:
idle_distance, idle_witness = before_distances["idle"]
display(idle_witness)
combined_fault = ec.FaultEvent()
for factor in idle_witness:
    combined_fault *= factor

(witness_effect,) = bare_idle_profile.effects_of([combined_fault])
print("Checks flipped:", sorted(witness_effect.syndrome))
print("Output logical Paulis:", dict(witness_effect.output_error))
assert idle_distance == len(idle_witness) == 1
assert not witness_effect.syndrome
assert any(error.weight for error in witness_effect.output_error.values())

[FaultEvent.after(3, Pauli('IIIIX'))]

Checks flipped: []
Output logical Paulis: {0: X}


## 5. Restore the idle distance

Use the self-checking C4 syndrome circuit from [Reichardt, page 4, Sec. II.2](https://arxiv.org/pdf/1804.06995#page=4). The X- and Z-syndrome couplings are interleaved so each ancilla detects dangerous faults from the other. The circuit uses eight CNOTs and the same two ancillas, with no additional flag qubit.

The paper's data qubits 1-4 become 0-3 here. Ancilla 4 measures the X stabilizer and ancilla 5 the Z stabilizer. We assume these couplings are available and omit the paper's geometric swaps. The model includes quantum errors and readout flips at the listed calls, but no additional movement or idle locations.

Build a replacement gadget, derive its checks, and compare its action with the original. Then assign the replacement mapping back to `layer.gadgets`: changing an entry in the returned dictionary alone would not update the layer.

In [11]:
import stim
stim.Circuit(bare_idle.circuit.source).diagram()

q0: -----X-----------@-------------------------
         |           |
q1: -----|-X---------|-@-----------------------
         | |         | |
q2: -----|-|-X-------|-|-@---------------------
         | | |       | | |
q3: -----|-|-|-X-----|-|-|-@-------------------
         | | | |     | | | |
q4: -R-H-@-@-@-@-H---|-|-|-|-M:rec[0]----------
                     | | | |
q5: -------------R-H-@-@-@-@-H--------M:rec[1]-

In [12]:
reichardt_source = """R 4 5
H 4
CX 4 0
CX 2 5
CX 0 5
CX 1 5
CX 4 2
CX 4 3
CX 4 1
CX 3 5
H 4
M 4 5
"""
stim.Circuit(reichardt_source).diagram()

q0: -----X---@----------------------
         |   |
q1: -----|---|-@-----X--------------
         |   | |     |
q2: -----|-@-|-|-X---|--------------
         | | | | |   |
q3: -----|-|-|-|-|-X-|-@------------
         | | | | | | | |
q4: -R-H-@-|-|-|-@-@-@-|-H-M:rec[0]-
           | | |       |
q5: -R-----X-X-X-------X---M:rec[1]-

In [13]:
self_checking_idle = ec.derive(qc.Gadget(
    bare_idle.implements,
    qc.gadgets.Circuit(bare_idle.circuit.instruction_set, reichardt_source, format="stim"),
    inputs=bare_idle.inputs,
    outputs=bare_idle.outputs,
))
assert isinstance(self_checking_idle, qc.Gadget)
self_checking_profile = ec.GadgetProfile(self_checking_idle)
assert self_checking_profile.is_equivalent_to(bare_idle_profile)
assert self_checking_idle.checks == bare_idle.checks
repaired_idle_distance, repaired_idle_witness = self_checking_profile.distance()
assert repaired_idle_distance == len(repaired_idle_witness) == code_distance

gadgets = layer.gadgets
gadgets["idle"] = self_checking_idle
layer.gadgets = gadgets
assert not ec.audit(protocol).diagnostics
print("Idle circuit distance:", idle_distance, "->", repaired_idle_distance)

Idle circuit distance: 1 -> 2


## 6. Repair both preparations

`prepare_z` resets the four data qubits before syndrome extraction. `prepare_x` also applies H to each data qubit. Both original preparations had distance one; fixing idle alone would leave those failures in the protocol.

Keep each preparation prefix and replace its independent syndrome rounds with the same self-checking circuit. Derive checks for the new preparation rather than copying idle's checks: preparations have no input encoding, so their boundary relations differ.

Verify that each replacement prepares the same logical state, and that its minimum undetected failure now needs two faults.

In [14]:
preparation_prefixes = {
    "prepare_z": "R 0 1 2 3\n",
    "prepare_x": "R 0 1 2 3\nH 0 1 2 3\n",
}
gadgets = layer.gadgets
for mnemonic, prefix in preparation_prefixes.items():
    previous = gadgets[mnemonic]
    replacement = ec.derive(qc.Gadget(
        previous.implements,
        qc.gadgets.Circuit(previous.circuit.instruction_set, prefix + reichardt_source, format="stim"),
        inputs=previous.inputs,
        outputs=previous.outputs,
    ))
    assert isinstance(replacement, qc.Gadget)
    replacement_profile = ec.GadgetProfile(replacement)
    assert replacement_profile.is_equivalent_to(ec.GadgetProfile(previous))
    preparation_distance, preparation_witness = replacement_profile.distance()
    assert preparation_distance == len(preparation_witness) == code_distance
    gadgets[mnemonic] = replacement
    print(mnemonic, "distance:", before_distances[mnemonic][0], "->", preparation_distance)
layer.gadgets = gadgets
assert not ec.audit(protocol).diagnostics

prepare_z distance: 1 -> 2


prepare_x distance: 1 -> 2


### How the ancillas check each other

The gate order makes a dangerous ancilla fault visible to the other ancilla:

- **X on ancilla 4 after `CX 4 2`:** the remaining couplings spread X to data qubits 3 and 1. Qubit 3 still has to couple to ancilla 5 through `CX 3 5`, so it flips the Z-syndrome result. Qubit 1 has already coupled to ancilla 5, so its X cannot cancel that flip.
- **Z on ancilla 5 after `CX 0 5`:** `CX 1 5` spreads Z backwards to data qubit 1, and the later `CX 4 1` carries it to ancilla 4. The final H turns that Z into X, flipping the X-syndrome result. The Z that reaches data qubit 3 arrives after `CX 4 3`, so it cannot cancel the ancilla error.

These are separate experiments, evaluated together by `effects_of`. In `Circuit.calls`, the injection gates are at positions 7 and 5. The first two assertions verify those positions before creating the faults.

The X-syndrome bit appears in checks 0 and 2; the Z-syndrome bit appears in checks 1 and 3. Each pair compares the same measurement with the input and output stabilizer signs. Thus `[1, 3]` means the Z-syndrome bit reveals the X fault, not that two independent measurements detected it.

These two examples explain the mechanism. The distance searches above test all allowed quantum and readout errors and establish that neither repaired idle nor either preparation has an undetected single-fault logical failure.

In [15]:
self_checking_calls = self_checking_idle.circuit.calls
assert (self_checking_calls[7].mnemonic, self_checking_calls[7].operands) == ("CX", [4, 2])
assert (self_checking_calls[5].mnemonic, self_checking_calls[5].operands) == ("CX", [0, 5])

x_hook = ec.FaultEvent.after(7, ec.Pauli("X_4"))
z_hook = ec.FaultEvent.after(5, ec.Pauli("Z_5"))
x_effect, z_effect = self_checking_profile.effects_of([x_hook, z_hook])

print("X on ancilla 4 is detected by Z checks:", sorted(x_effect.syndrome))
print("Z on ancilla 5 is detected by X checks:", sorted(z_effect.syndrome))
assert x_effect.syndrome == {1, 3}
assert z_effect.syndrome == {0, 2}

X on ancilla 4 is detected by Z checks: [1, 3]
Z on ancilla 5 is detected by X checks: [0, 2]


## 7. Check the whole revised qodec

Re-run the audit and every gadget's distance search on the protocol's installed gadgets. Do not rely only on a replacement object that might never have been attached to the layer.

The final menu still has both preparations, both destructive measurements, idle, and transversal CNOT. All six gadgets have distance two under the quantum and readout-fault model, with a two-event witness for each.

Require **no diagnostics**, rather than only `report.ok`: the latter allows warnings. These checks establish noiseless consistency and the stated gadget-distance property separately.

In [16]:
protocol.description = (
    "C4 built with bare-css/v1, with repaired frame equations "
    "and self-checking idle and preparation circuits."
)
final_report = ec.audit(protocol)
final_distances = {
    mnemonic: ec.GadgetProfile(gadget).distance()
    for mnemonic, gadget in protocol.layers[0].gadgets.items()
}
assert not final_report.diagnostics
assert set(protocol.layers[0].instruction_set.instructions) == set(final_distances) == instruction_menu
assert all(
    gadget_distance == len(witness) == code_distance
    for gadget_distance, witness in final_distances.values()
)
display(Markdown("\n".join([
    "| Gadget | Before circuit repair | Final distance |",
    "| --- | ---: | ---: |",
    *(f"| `{mnemonic}` | {before_distances[mnemonic][0]} | {gadget_distance} |"
      for mnemonic, (gadget_distance, witness) in sorted(final_distances.items())),
])))
print(final_report)

| Gadget | Before circuit repair | Final distance |
| --- | ---: | ---: |
| `idle` | 1 | 2 |
| `measure_x` | 2 | 2 |
| `measure_z` | 2 | 2 |
| `prepare_x` | 1 | 2 |
| `prepare_z` | 1 | 2 |
| `transversal_cx` | 2 | 2 |

audit: ok (no diagnostics)


## 8. Save and check the result

A qodec carries the revised circuits and equations as ordinary data. `save` takes a directory; with `single_file=True`, the bundle is written there under `manifest_filename`. This example uses a temporary directory and keeps the reloaded qodec in memory.

Compare the complete declarations after reloading, then audit and measure every gadget again. Serialization preserves a design; it does not prove the design correct.

In [19]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    protocol.save(directory, single_file=True)
    saved_path = Path(directory) / protocol.manifest_filename
    reloaded = qc.Qodec.load(saved_path)
print("Round trip:", reloaded == protocol)

Round trip: True


## What we established

Starting only from the C4 code, we built a bare qodec, inspected its six retained instructions, and repaired all audit findings without dropping functionality. Then we found and repaired distance-one faults in idle and both preparations.

The final `protocol` and `reloaded` qodecs have no audit errors, warnings, or informational findings. All six gadgets have distance two, including both destructive measurements, with quantum and recorded-bit errors included.

This is a gadget-level result under the stated Pauli and readout-fault model, not a certificate for every hardware implementation or composition. Before using the protocol with a device, account for its connectivity, timing, additional fault locations, and noise correlations.